In [ ]:
# ============================================================
# PATH CONFIGURATION — Adjust for your environment
# ============================================================
# These notebooks were originally developed on Google Colab.
# Paths referencing /content/drive/MyDrive/... are Colab-specific.
#
# For local execution, set ROOT_DIR to your project root:
#
#   import os
#   ROOT_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
#   # Then replace /content/drive/MyDrive/... paths with os.path.join(ROOT_DIR, ...)
#
# The authoritative dataset files are in: data/train.parquet, data/validation.parquet, data/test.parquet
# Images are in: data/images/
# ============================================================


# MODULE E — BENCHMARK DATASET BUILDER
**Single responsibility:** Merge benign + malicious datasets → ML-ready benchmark with group-aware splits.

```
MASTER_BENIGN_DATASET
        +
FINAL_DATASET (malicious)
        ↓
Schema Standardization
        ↓
Merge
        ↓
Group-aware Train/Val/Test Split
        ↓
FINAL_BENCHMARK_DATASET/
```

In [17]:
# ── CELL 0 : Install / imports ───────────────────────────────────────────────
!pip install pyarrow fastparquet -q

import os, json, hashlib, logging
from pathlib import Path
from datetime import datetime

import numpy  as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from google.colab import drive

drive.mount('/content/drive')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s │ %(levelname)s │ %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('MODULE_E')
print('✅ Imports OK')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Imports OK


In [18]:
# ── CELL 1 : PATHS — edit these to match your Drive layout ───────────────────
DRIVE_BASE = Path('/content/drive/MyDrive')   # ← adjust if different

# Source datasets
BENIGN_TEXT_CSV   = DRIVE_BASE / 'MASTER_BENIGN_DATASET/TEXT_SOURCES/benign_text.csv'
BENIGN_IMAGE_META = DRIVE_BASE / 'MASTER_BENIGN_DATASET/TEXT_SOURCES/image_metadata.csv'
BENIGN_IMAGE_DIR  = DRIVE_BASE / 'MASTER_BENIGN_DATASET/TEXT_SOURCES/benign_images'

MALICIOUS_PARQUET = DRIVE_BASE / 'FINAL_DATASET/PARQUET/malicious_dataset.parquet'

# Output
OUT_DIR = DRIVE_BASE / 'FINAL_BENCHMARK_DATASET'
OUT_DIR.mkdir(parents=True, exist_ok=True)
for sub in ['TEXT', 'IMAGES', 'LABELS']:
    (OUT_DIR / sub).mkdir(exist_ok=True)

# Split ratios
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
RANDOM_SEED = 42

print(f'Output → {OUT_DIR}')

Output → /content/drive/MyDrive/FINAL_BENCHMARK_DATASET


In [47]:
# ── CELL 2 : STEP 1 — Load & Standardize Benign dataset ──────────────────────
#
# Source schema:
#   benign_text.csv  → source_id, filename, source_dataset,
#                       extraction_method, page_count, extracted_text, extraction_status
#   image_metadata.csv → image_id, image_path, ocr_text, source_dataset, label
#
# Target schema (mirrors malicious):
#   sample_id, source_benign_id, text, label,
#   attack_family, attack_name, attack_location, attack_type,
#   visibility, severity, ocr_survivability, target_component,
#   payload_id, image_path, runtime_ocr

log.info('Loading benign text CSV …')
df_btext = pd.read_csv(BENIGN_TEXT_CSV, low_memory=False)
log.info(f'  benign_text rows : {len(df_btext):,}')
print(df_bimg["image_path"].iloc[0])

print(
    df_bimg["image_path"]
    .astype(str)
    .str.replace(
        "/content/drive/MyDrive/TEXT_SOURCES/",
        "/content/drive/MyDrive/MASTER_BENIGN_DATASET/TEXT_SOURCES/",
        regex=False
    )
    .iloc[0]
)

log.info('Loading benign image metadata …')
df_bimg = pd.read_csv(BENIGN_IMAGE_META, low_memory=False)
log.info(f'  image_metadata rows : {len(df_bimg):,}')

# ── 2a. Standardize text-sourced benign rows ──────────────────────────────────
# Only keep successfully extracted rows
df_btext_ok = df_btext[df_btext['extraction_status'].str.lower().str.contains('success', na=False)].copy()
log.info(f'  Benign text rows after status filter : {len(df_btext_ok):,}')

benign_text_std = pd.DataFrame({
    'source_benign_id'   : df_btext_ok['source_id'].values,
    'text'               : df_btext_ok['extracted_text'].values,
    'label'              : 'BENIGN',
    'source_dataset'     : df_btext_ok['source_dataset'].values,
    'image_path'         : None,          # text-sourced → no image
    'runtime_ocr'        : df_btext_ok['extracted_text'].values,  # same as text
})

# ── 2b. Standardize image-sourced benign rows ─────────────────────────────────
benign_img_std = pd.DataFrame({
    'source_benign_id'   : df_bimg['image_id'].values,
    'text'               : df_bimg['ocr_text'].values,
    'label'              : 'BENIGN',
    'source_dataset'     : df_bimg['source_dataset'].values,
    'image_path' : (

    df_bimg["image_path"]

    .astype(str)

    .str.replace(

        "/content/drive/MyDrive/TEXT_SOURCES/",

        "/content/drive/MyDrive/MASTER_BENIGN_DATASET/TEXT_SOURCES/",

        regex=False

    )

),
    'runtime_ocr'        : df_bimg['ocr_text'].values,
})

# ── 2c. Concatenate both benign sources ───────────────────────────────────────
df_benign_raw = pd.concat([benign_text_std, benign_img_std], ignore_index=True)
df_benign_raw = df_benign_raw.dropna(subset=['text'])          # drop empty text
df_benign_raw = df_benign_raw[df_benign_raw['text'].str.strip().str.len() > 10]  # min chars
df_benign_raw = df_benign_raw.reset_index(drop=True)
log.info(f'  Combined benign rows (after clean) : {len(df_benign_raw):,}')

# ── 2d. Add NULL attack columns ───────────────────────────────────────────────
ATTACK_NULL_COLS = [
    'attack_family', 'attack_name', 'attack_location', 'attack_type',
    'visibility', 'severity', 'ocr_survivability', 'target_component',
    'payload_id', 'generation_timestamp',
]
for col in ATTACK_NULL_COLS:
    df_benign_raw[col] = None

# ── 2e. Assign sample_id  B_000001 … ─────────────────────────────────────────
df_benign_raw.insert(0, 'sample_id',
    ['B_{:06d}'.format(i+1) for i in range(len(df_benign_raw))])

print(df_benign_raw.head(3))
print(f'\n✅ BENIGN standardized : {len(df_benign_raw):,} rows')
print(benign_img_std["image_path"].head())

/content/drive/MyDrive/TEXT_SOURCES/benign_images/IMG_00024.jpg
/content/drive/MyDrive/MASTER_BENIGN_DATASET/TEXT_SOURCES/benign_images/IMG_00024.jpg
  sample_id source_benign_id  \
0  B_000001        CSV_00000   
1  B_000002        CSV_00001   
2  B_000003        CSV_00002   

                                                text   label source_dataset  \
0  Filename: med_doc_discharge_summary_200001_noi...  BENIGN  INSURANCE_CSV   
1  Filename: med_doc_discharge_summary_200002_noi...  BENIGN  INSURANCE_CSV   
2  Filename: med_doc_discharge_summary_200003_noi...  BENIGN  INSURANCE_CSV   

  image_path                                        runtime_ocr attack_family  \
0       None  Filename: med_doc_discharge_summary_200001_noi...          None   
1       None  Filename: med_doc_discharge_summary_200002_noi...          None   
2       None  Filename: med_doc_discharge_summary_200003_noi...          None   

  attack_name attack_location attack_type visibility severity  \
0        None 

In [46]:
print(df_bimg["image_path"].head())

0    /content/drive/MyDrive/TEXT_SOURCES/benign_ima...
1    /content/drive/MyDrive/TEXT_SOURCES/benign_ima...
2    /content/drive/MyDrive/TEXT_SOURCES/benign_ima...
3    /content/drive/MyDrive/TEXT_SOURCES/benign_ima...
4    /content/drive/MyDrive/TEXT_SOURCES/benign_ima...
Name: image_path, dtype: object


In [48]:
# ── CELL 3 : Save BENIGN_STANDARDIZED.parquet ────────────────────────────────
BENIGN_STD_PATH = OUT_DIR / 'BENIGN_STANDARDIZED.parquet'
df_benign_raw.to_parquet(BENIGN_STD_PATH, index=False)
log.info(f'Saved → {BENIGN_STD_PATH}  ({BENIGN_STD_PATH.stat().st_size/1e6:.1f} MB)')

In [49]:
# ── CELL 4 : STEP 2 — Load & Standardize Malicious dataset ───────────────────
#
# malicious_dataset.parquet columns (23):
#   malicious_id, source_benign_id, source_text_hash, payload_id,
#   attack_family, attack_name, attack_location, attack_type,
#   visibility, severity, ocr_survivability, target_component,
#   label, generation_timestamp, malicious_text,
#   image_path, image_width, image_height, file_checksum,
#   status, error, image_generation_timestamp, exists

log.info('Loading malicious parquet …')
df_mal = pd.read_parquet(MALICIOUS_PARQUET)
log.info(f'  Malicious rows : {len(df_mal):,}  | cols : {df_mal.shape[1]}')

# Rename to unified schema
df_mal_std = df_mal.rename(columns={
    'malicious_id'   : 'sample_id',
    'malicious_text' : 'text',
}).copy()

# Ensure label column is uppercase string
df_mal_std['label'] = df_mal_std['label'].astype(str).str.upper().fillna('MALICIOUS')

# Add source_dataset tag (not present in malicious)
df_mal_std['source_dataset'] = 'GENERATED_MALICIOUS'

# runtime_ocr: use the malicious text itself as reference
if 'runtime_ocr' not in df_mal_std.columns:
    df_mal_std['runtime_ocr'] = df_mal_std['text']

print(df_mal_std[['sample_id','source_benign_id','label','attack_family','severity']].head(3))
print(f'\n✅ MALICIOUS standardized : {len(df_mal_std):,} rows')

              sample_id source_benign_id label      attack_family severity
0  MAL-TXT-007ef62644b9        IMG_02242     1  Context Poisoning     high
1  MAL-TXT-00ae00116d96        CSV_71104     1  Context Poisoning     high
2  MAL-TXT-00b4b7eb51dc       CSV_116689     1  Context Poisoning     high

✅ MALICIOUS standardized : 144,937 rows


In [50]:
# ── CELL 5 : STEP 3 — Align schemas then MERGE ───────────────────────────────

# Collect all columns across both frames
all_cols = list(dict.fromkeys(
    list(df_benign_raw.columns) + list(df_mal_std.columns)
))

# Reindex both to full column set (missing cols become NaN)
df_b_aligned = df_benign_raw.reindex(columns=all_cols)
df_m_aligned = df_mal_std.reindex(columns=all_cols)

# Merge
df_full = pd.concat([df_b_aligned, df_m_aligned], ignore_index=True)

# Normalise label
df_full['label'] = df_full['label'].str.upper()

# Sanity check
vc = df_full['label'].value_counts()
log.info(f'FULL_DATASET rows : {len(df_full):,}')
log.info(f'Label distribution :\n{vc.to_string()}')
print(f'\nColumns ({len(df_full.columns)}) : {list(df_full.columns)}')

/tmp/ipykernel_2341/856483410.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_full = pd.concat([df_b_aligned, df_m_aligned], ignore_index=True)
/tmp/ipykernel_2341/856483410.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_full = pd.concat([df_b_aligned, df_m_aligned], ignore_index=True)
/tmp/ipykernel_2341/856483410.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or al


Columns (25) : ['sample_id', 'source_benign_id', 'text', 'label', 'source_dataset', 'image_path', 'runtime_ocr', 'attack_family', 'attack_name', 'attack_location', 'attack_type', 'visibility', 'severity', 'ocr_survivability', 'target_component', 'payload_id', 'generation_timestamp', 'source_text_hash', 'image_width', 'image_height', 'file_checksum', 'status', 'error', 'image_generation_timestamp', 'exists']


In [56]:
from pathlib import Path

bad = df_full[
    df_full["label"].eq("BENIGN")
    &
    df_full["image_path"].notna()
    &
    (df_full["image_path"].astype(str).str.strip() != "")
]

bad = bad[
    ~bad["image_path"].apply(lambda x: Path(x).exists())
]

print("Broken benign image paths :", len(bad))

display(
    bad[
        ["sample_id", "source_benign_id", "image_path"]
    ].head(20)
)

Broken benign image paths : 0


,sample_id,source_benign_id,image_path


In [51]:
# ── CELL 6 : STEP 4 — Drop existing split column (if present) ────────────────
#
# FINAL_DATASET SPLITS/ only split malicious samples.
# We delete that and re-split the full merged data correctly.

if 'split' in df_full.columns:
    df_full = df_full.drop(columns=['split'])
    log.info('Dropped existing `split` column → will re-split correctly.')
else:
    log.info('No existing `split` column found (OK).')

print('✅ Step 4 done')

✅ Step 4 done


In [52]:
# ── CELL 7 : STEP 5 — Group-aware Train / Val / Test split ───────────────────
#
# Groups = source_benign_id
# This guarantees that all injected variants of a document
# stay in the SAME split → prevents data leakage.
#
# Strategy:
#   GroupShuffleSplit  80 % train  |  20 % temp
#   GroupShuffleSplit  50 % val    |  50 % test   (of the 20 % temp)

log.info('Running GroupShuffleSplit …')

groups = df_full['source_benign_id'].fillna(df_full['sample_id'])  # benign rows ARE their own group

# ── Split 1 : train vs temp (val+test) ───────────────────────────────────────
gss1 = GroupShuffleSplit(n_splits=1, test_size=(VAL_RATIO + TEST_RATIO), random_state=RANDOM_SEED)
train_idx, temp_idx = next(gss1.split(df_full, groups=groups))

df_train = df_full.iloc[train_idx].copy()
df_temp  = df_full.iloc[temp_idx].copy()
groups_temp = groups.iloc[temp_idx]

# ── Split 2 : val vs test (within temp) ──────────────────────────────────────
val_frac = VAL_RATIO / (VAL_RATIO + TEST_RATIO)   # 0.5
gss2 = GroupShuffleSplit(n_splits=1, test_size=(1 - val_frac), random_state=RANDOM_SEED)
val_idx, test_idx = next(gss2.split(df_temp, groups=groups_temp))

df_val  = df_temp.iloc[val_idx].copy()
df_test = df_temp.iloc[test_idx].copy()

# Tag split column
df_train['split'] = 'train'
df_val['split']   = 'validation'
df_test['split']  = 'test'

# ── Verify each split has BOTH labels ─────────────────────────────────────────
for name, df_s in [('TRAIN', df_train), ('VAL', df_val), ('TEST', df_test)]:
    vc = df_s['label'].value_counts()
    log.info(f'{name:5s} | {len(df_s):7,} rows | {vc.to_dict()}')

print('\n✅ Group-aware split done — no data leakage.')


✅ Group-aware split done — no data leakage.


In [53]:
# ── CELL 8 : STEP 6 — Save FINAL_BENCHMARK_DATASET ───────────────────────────

# ── 8a. Parquet splits ────────────────────────────────────────────────────────
log.info('Writing parquet splits …')
df_train.to_parquet(OUT_DIR / 'train.parquet',      index=False)
df_val.to_parquet(  OUT_DIR / 'validation.parquet', index=False)
df_test.to_parquet( OUT_DIR / 'test.parquet',       index=False)

# ── 8b. metadata.csv (full dataset) ──────────────────────────────────────────
log.info('Writing metadata.csv …')
df_full_tagged = pd.concat([df_train, df_val, df_test], ignore_index=True)
df_full_tagged.to_csv(OUT_DIR / 'metadata.csv', index=False)

# ── 8c. TEXT/  — text content only ───────────────────────────────────────────
text_cols = ['sample_id', 'source_benign_id', 'label', 'split', 'text', 'runtime_ocr', 'source_dataset']
df_full_tagged[text_cols].to_csv(OUT_DIR / 'TEXT' / 'benchmark_text.csv', index=False)

# ── 8d. LABELS/ — label metadata ─────────────────────────────────────────────
label_cols = [
    'sample_id', 'source_benign_id', 'label', 'split',
    'attack_family', 'attack_name', 'attack_location', 'attack_type',
    'visibility', 'severity', 'ocr_survivability', 'target_component', 'payload_id'
]
df_full_tagged[[c for c in label_cols if c in df_full_tagged.columns]].to_csv(
    OUT_DIR / 'LABELS' / 'benchmark_labels.csv', index=False
)

log.info('All files written.')
print('✅ Parquet splits + CSVs saved.')

✅ Parquet splits + CSVs saved.


In [55]:
from pathlib import Path

p = Path(
    benign_img_std["image_path"].iloc[0]
)

print(p)
print("Exists :", p.exists())

/content/drive/MyDrive/MASTER_BENIGN_DATASET/TEXT_SOURCES/benign_images/IMG_00024.jpg
Exists : True


In [57]:
# ── CELL 8b : STEP 6c — Populate IMAGES/ and update image_path metadata ─────
#
# 1. Copy benign images  → FINAL_BENCHMARK_DATASET/IMAGES/
# 2. Copy malicious images → FINAL_BENCHMARK_DATASET/IMAGES/
# 3. Skip if file already exists (no overwrite)
# 4. Rewrite every non-empty image_path in df_full_tagged to point at IMAGES/
# 5. Re-save metadata.csv, parquet splits, and label CSV with updated paths
# 6. Verify every non-empty image_path exists on disk
# 7. Print copy summary

import shutil

IMAGES_DIR = OUT_DIR / 'IMAGES'
IMAGES_DIR.mkdir(exist_ok=True)

benign_copied   = 0
malicious_copied = 0
skipped         = 0

# ── helper: copy one image, return ('copied'|'skipped'|'missing') ─────────────
def _copy_image(src_path_str):
    """Copy src → IMAGES_DIR/<filename>. Returns status string."""
    if not src_path_str or (isinstance(src_path_str, float)):
        return 'skip_null'
    src = Path(str(src_path_str))
    if not src.is_file():
        return 'missing'
    dst = IMAGES_DIR / src.name
    if dst.exists():
        return 'skipped'
    shutil.copy2(src, dst)
    return 'copied'

# ── 1. Copy benign images ─────────────────────────────────────────────────────
log.info('Copying benign images …')
benign_img_mask = (
    (df_full_tagged['label'] == 'BENIGN') &
    df_full_tagged['image_path'].notna() &
    (df_full_tagged['image_path'].astype(str).str.strip() != '') &
    (df_full_tagged['image_path'].astype(str).str.lower() != 'none')
)
benign_img_paths = df_full_tagged.loc[benign_img_mask, 'image_path'].unique()

for p in benign_img_paths:
    status = _copy_image(p)
    if status == 'copied':  benign_copied += 1
    elif status == 'skipped': skipped += 1
    elif status == 'missing': log.warning(f'Benign image not found on disk: {p}')

# ── 2. Copy malicious images ──────────────────────────────────────────────────
log.info('Copying malicious images …')
mal_img_mask = (
    (df_full_tagged['label'] != 'BENIGN') &
    df_full_tagged['image_path'].notna() &
    (df_full_tagged['image_path'].astype(str).str.strip() != '') &
    (df_full_tagged['image_path'].astype(str).str.lower() != 'none')
)
mal_img_paths = df_full_tagged.loc[mal_img_mask, 'image_path'].unique()

for p in mal_img_paths:
    status = _copy_image(p)
    if status == 'copied':  malicious_copied += 1
    elif status == 'skipped': skipped += 1
    elif status == 'missing': log.warning(f'Malicious image not found on disk: {p}')

# ── 3. Rewrite image_path → IMAGES/<filename> in df_full_tagged ───────────────
log.info('Rewriting image_path to point at IMAGES/ …')

def _remap_image_path(val):

    if pd.isna(val):
        return ""

    val = str(val).strip()

    if val in ("", "None", "nan"):
        return ""

    dst = IMAGES_DIR / Path(val).name

    if dst.exists():
        return str(dst)

    return ""

df_full_tagged["image_path"] = (
    df_full_tagged["image_path"]
    .apply(_remap_image_path)
)

df_full_tagged["exists"] = (
    df_full_tagged["image_path"]
    .apply(lambda x: Path(x).exists() if str(x).strip() else False)
)

# Propagate updated image_path back into the split frames
updated_ip = df_full_tagged.set_index('sample_id')['image_path']
updated_exists = (
    df_full_tagged
    .set_index("sample_id")["exists"]
)
for df_split in [df_train, df_val, df_test]:
    df_split['image_path'] = df_split['sample_id'].map(updated_ip)


# ── 4. Re-save files that contain image_path ──────────────────────────────────
log.info('Re-saving metadata.csv and parquet splits with updated image_path …')
df_full_tagged.to_csv(OUT_DIR / 'metadata.csv', index=False)
df_train.to_parquet(OUT_DIR / 'train.parquet',      index=False)
df_val.to_parquet(  OUT_DIR / 'validation.parquet', index=False)
df_test.to_parquet( OUT_DIR / 'test.parquet',       index=False)

# Also re-save benchmark_labels.csv (contains image_path indirectly via metadata)
label_cols_img = [
    'sample_id', 'source_benign_id', 'label', 'split',
    'attack_family', 'attack_name', 'attack_location', 'attack_type',
    'visibility', 'severity', 'ocr_survivability', 'target_component', 'payload_id'
]
df_full_tagged[[c for c in label_cols_img if c in df_full_tagged.columns]].to_csv(
    OUT_DIR / 'LABELS' / 'benchmark_labels.csv', index=False
)

# ── 5. Verify every non-empty image_path exists on disk ──────────────────────
log.info('Verifying image_path integrity …')
img_rows = df_full_tagged[
    df_full_tagged['image_path'].notna() &
    (df_full_tagged['image_path'].astype(str).str.strip() != '') &
    (df_full_tagged['image_path'].astype(str).str.lower().isin(['none','nan']) == False)
]
missing_on_disk = []
for _, row in img_rows.iterrows():
    if not Path(str(row['image_path'])).is_file():
        missing_on_disk.append(row['image_path'])

if missing_on_disk:
    log.warning(f'{len(missing_on_disk)} image_path entries missing on disk (source files not found at copy time):')
    for p in missing_on_disk[:10]:
        log.warning(f'  MISSING: {p}')
else:
    log.info('All non-empty image_path entries verified on disk ✅')

# ── 6. Summary ───────────────────────────────────────────────────────────────
total_in_images_dir = len([
    f
    for f in IMAGES_DIR.iterdir()
    if f.is_file()
])

print('\n' + '='*60)
print('IMAGE COPY SUMMARY')
print('='*60)
print(f'  Benign images copied         : {benign_copied}')
print(f'  Malicious images copied      : {malicious_copied}')
print(f'  Images skipped (exist)       : {skipped}')
print(f'  Total images in IMAGES/      : {total_in_images_dir}')
if missing_on_disk:
    print(f'  ⚠ image_path missing on disk : {len(missing_on_disk)}')
else:
    print(f'  image_path verification      : ALL OK ✅')
print('='*60)
print('✅ IMAGES/ populated and image_path metadata updated.')



IMAGE COPY SUMMARY
  Benign images copied         : 0
  Malicious images copied      : 0
  Images skipped (exist)       : 7303
  Total images in IMAGES/      : 7303
  image_path verification      : ALL OK ✅
✅ IMAGES/ populated and image_path metadata updated.


In [58]:
print("Broken:", (
    df_full_tagged["image_path"]
    .fillna("")
    .ne("")
    &
    (~df_full_tagged["image_path"].apply(lambda x: Path(x).exists()))
).sum())

Broken: 0


In [59]:
bad = df_full_tagged[
    df_full_tagged["image_path"].fillna("").ne("")
    &
    (~df_full_tagged["image_path"].apply(lambda x: Path(x).exists()))
]

display(
    bad[
        ["sample_id","label","image_path"]
    ].head(20)
)

,sample_id,label,image_path


In [28]:
# ── CELL 9 : STEP 6b — statistics.json ───────────────────────────────────────

def split_stats(df_s, name):
    vc   = df_s['label'].value_counts().to_dict()
    af   = df_s['attack_family'].value_counts().dropna().to_dict()
    grps = df_s['source_benign_id'].fillna(df_s['sample_id']).nunique()
    return {
        'rows'           : int(len(df_s)),
        'unique_groups'  : int(grps),
        'label_dist'     : {str(k): int(v) for k, v in vc.items()},
        'attack_families': {str(k): int(v) for k, v in af.items()},
    }

stats = {
    'module'              : 'MODULE_E',
    'generated_at'        : datetime.now().isoformat(),
    'dataset_version'     : 'v1.0',
    'total_rows'          : int(len(df_full_tagged)),
    'total_benign'        : int((df_full_tagged['label'] == 'BENIGN').sum()),
    'total_malicious'     : int((df_full_tagged['label'] != 'BENIGN').sum()),
    'split_ratios'        : {'train': TRAIN_RATIO, 'validation': VAL_RATIO, 'test': TEST_RATIO},
    'group_key'           : 'source_benign_id',
    'random_seed'         : RANDOM_SEED,
    'leakage_prevention'  : 'GroupShuffleSplit on source_benign_id',
    'splits'              : {
        'train'      : split_stats(df_train, 'train'),
        'validation' : split_stats(df_val,   'validation'),
        'test'       : split_stats(df_test,  'test'),
    },
    'schema_columns'      : list(df_full_tagged.columns),
}

with open(OUT_DIR / 'statistics.json', 'w') as f:
    json.dump(stats, f, indent=2)

print(json.dumps({k: v for k, v in stats.items() if k != 'schema_columns'}, indent=2))
print('\n✅ statistics.json saved.')

{
  "module": "MODULE_E",
  "generated_at": "2026-06-25T14:18:11.493465",
  "dataset_version": "v1.0",
  "total_rows": 293411,
  "total_benign": 148474,
  "total_malicious": 144937,
  "split_ratios": {
    "train": 0.8,
    "validation": 0.1,
    "test": 0.1
  },
  "group_key": "source_benign_id",
  "random_seed": 42,
  "leakage_prevention": "GroupShuffleSplit on source_benign_id",
  "splits": {
    "train": {
      "rows": 234782,
      "unique_groups": 115956,
      "label_dist": {
        "BENIGN": 118834,
        "1": 115948
      },
      "attack_families": {
        "OCR-specific Attacks": 7773,
        "Header Injection": 7762,
        "Insurance-specific Attacks": 7751,
        "Data Exfiltration": 7749,
        "White Text Injection": 7746,
        "Tiny Font Injection": 7733,
        "Watermark Injection": 7733,
        "Instruction Override": 7732,
        "Policy Bypass": 7726,
        "Invisible Character Injection": 7722,
        "Hidden Multimodal": 7722,
        "Contex

In [29]:
# ── CELL 10 : Quick sanity checks ────────────────────────────────────────────

print('=' * 60)
print('FINAL BENCHMARK DATASET — SANITY CHECKS')
print('=' * 60)

checks = []

# 1. No group leakage between train and test
train_groups = set(df_train['source_benign_id'].dropna())
test_groups  = set(df_test['source_benign_id'].dropna())
leak = train_groups & test_groups
status = 'PASS' if len(leak) == 0 else f'FAIL — {len(leak)} leaked groups'
checks.append({'check': 'No group leakage train↔test', 'status': status})
print(f'[{status}] No group leakage train↔test')

# 2. Both labels in every split
for name, df_s in [('train', df_train), ('validation', df_val), ('test', df_test)]:
    labels = df_s['label'].unique()
    has_both = ('BENIGN' in labels) and any(l != 'BENIGN' for l in labels)
    st = 'PASS' if has_both else 'FAIL — missing a label'
    checks.append({'check': f'Both labels in {name}', 'status': st})
    print(f'[{st}] Both labels in {name} : {sorted(labels)}')

# 3. No null text in any split
for name, df_s in [('train', df_train), ('validation', df_val), ('test', df_test)]:
    nulls = df_s['text'].isna().sum()
    st = 'PASS' if nulls == 0 else f'WARN — {nulls} null texts'
    checks.append({'check': f'No null text in {name}', 'status': st})
    print(f'[{st}] Null texts in {name} : {nulls}')

# 4. Total count consistent
total_splits = len(df_train) + len(df_val) + len(df_test)
st = 'PASS' if total_splits == len(df_full_tagged) else f'FAIL — mismatch {total_splits} vs {len(df_full_tagged)}'
checks.append({'check': 'Row count consistent', 'status': st})
print(f'[{st}] Row count : {total_splits:,}')

# Save validation report
pd.DataFrame(checks).to_csv(OUT_DIR / 'validation_report.csv', index=False)
print('\n✅ validation_report.csv saved.')

FINAL BENCHMARK DATASET — SANITY CHECKS
[PASS] No group leakage train↔test
[PASS] Both labels in train : ['1', 'BENIGN']
[PASS] Both labels in validation : ['1', 'BENIGN']
[PASS] Both labels in test : ['1', 'BENIGN']
[PASS] Null texts in train : 0
[PASS] Null texts in validation : 0
[PASS] Null texts in test : 0
[PASS] Row count : 293,411

✅ validation_report.csv saved.


In [30]:
# ── CELL 11 : Final folder tree ───────────────────────────────────────────────
import subprocess

def print_tree(path, prefix=''):
    entries = sorted(Path(path).iterdir())
    for i, entry in enumerate(entries):
        connector = '└── ' if i == len(entries)-1 else '├── '
        size = f'({entry.stat().st_size/1e6:.1f} MB)' if entry.is_file() else ''
        print(f'{prefix}{connector}{entry.name} {size}')
        if entry.is_dir():
            ext = '    ' if i == len(entries)-1 else '│   '
            print_tree(entry, prefix + ext)

print(f'\n📁 {OUT_DIR.name}/')
print_tree(OUT_DIR)

print('\n' + '='*60)
print('MODULE E COMPLETE ✅')
print('='*60)
print(f'  Train      : {len(df_train):>8,} rows')
print(f'  Validation : {len(df_val):>8,} rows')
print(f'  Test       : {len(df_test):>8,} rows')
print(f'  TOTAL      : {len(df_full_tagged):>8,} rows')
print('\nNext → Module F : Prompt Agent training on train.parquet')

Streaming output truncated to the last 5000 lines.
│   ├── IMG_02357.png (0.1 MB)
│   ├── IMG_02358.png (0.1 MB)
│   ├── IMG_02359.png (0.1 MB)
│   ├── IMG_02360.png (0.1 MB)
│   ├── IMG_02361.png (0.2 MB)
│   ├── IMG_02362.png (0.1 MB)
│   ├── IMG_02363.png (0.1 MB)
│   ├── IMG_02364.png (0.1 MB)
│   ├── IMG_02365.png (0.2 MB)
│   ├── IMG_02366.png (0.2 MB)
│   ├── IMG_02367.png (0.2 MB)
│   ├── IMG_02369.png (0.1 MB)
│   ├── IMG_02370.png (0.1 MB)
│   ├── IMG_02371.png (0.1 MB)
│   ├── IMG_02372.png (0.1 MB)
│   ├── IMG_02373.png (0.2 MB)
│   ├── IMG_02374.png (0.2 MB)
│   ├── IMG_02375.png (0.1 MB)
│   ├── IMG_02376.png (0.1 MB)
│   ├── IMG_02377.png (0.2 MB)
│   ├── IMG_02378.png (0.2 MB)
│   ├── IMG_02379.png (0.0 MB)
│   ├── IMG_02380.png (0.1 MB)
│   ├── IMG_02381.png (0.2 MB)
│   ├── IMG_02382.png (0.1 MB)
│   ├── IMG_02383.png (0.2 MB)
│   ├── IMG_02384.png (0.1 MB)
│   ├── IMG_02385.png (0.1 MB)
│   ├── IMG_02386.png (0.1 MB)
│   ├── IMG_02387.png (0.2 MB)
│   ├── IMG_02388.p

In [31]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
from collections import Counter

ROOT = "/content/drive/MyDrive/FINAL_BENCHMARK_DATASET"

print("="*100)
print("FINAL BENCHMARK DATA AUDIT")
print("="*100)

total_size = 0
total_files = 0

for root, dirs, files in os.walk(ROOT):

    print("\n")
    print("="*90)
    print(f"FOLDER : {os.path.relpath(root, ROOT)}")
    print("="*90)

    counter = Counter()
    folder_size = 0

    csv_files = []
    img_count = 0
    pdf_count = 0
    parquet_count = 0
    json_count = 0
    txt_count = 0

    for f in files:

        fp = os.path.join(root, f)

        try:
            size = os.path.getsize(fp)
        except:
            size = 0

        folder_size += size
        total_size += size
        total_files += 1

        ext = os.path.splitext(f)[1].lower()
        counter[ext] += 1

        if ext == ".csv":
            csv_files.append(fp)

        elif ext in [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"]:
            img_count += 1

        elif ext == ".pdf":
            pdf_count += 1

        elif ext == ".parquet":
            parquet_count += 1

        elif ext == ".json":
            json_count += 1

        elif ext in [".txt", ".text"]:
            txt_count += 1

    print(f"Subfolders  : {len(dirs)}")
    print(f"Files       : {len(files)}")
    print(f"Folder Size : {folder_size/(1024**2):.2f} MB")

    print("\nFile Types")
    for k, v in sorted(counter.items()):
        print(f"{k if k else '[no_ext]':12s} : {v}")

    print("\nSummary")
    print(f"Images   : {img_count}")
    print(f"PDF      : {pdf_count}")
    print(f"CSV      : {len(csv_files)}")
    print(f"Parquet  : {parquet_count}")
    print(f"JSON     : {json_count}")
    print(f"TXT      : {txt_count}")

    if files:
        print("\nSample Files")
        for f in files[:10]:
            print(" -", f)

    if csv_files:
        print("\nCSV DETAILS")

        for csv in csv_files:

            print("-"*70)
            print(os.path.basename(csv))

            try:
                df = pd.read_csv(csv, nrows=5)

                print("Rows Loaded :", len(df))
                print("Columns     :", len(df.columns))
                print("Column Names:")

                for c in df.columns:
                    print("  ", c)

            except Exception as e:
                print("Cannot read CSV :", e)

print("\n")
print("="*100)
print("FINAL BENCHMARK DATA SUMMARY")
print("="*100)

print(f"Total Files : {total_files}")
print(f"Total Size  : {total_size/(1024**3):.2f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FINAL BENCHMARK DATA AUDIT


FOLDER : .
Subfolders  : 3
Files       : 7
Folder Size : 565.86 MB

File Types
.csv         : 2
.json        : 1
.parquet     : 4

Summary
Images   : 0
PDF      : 0
CSV      : 2
Parquet  : 4
JSON     : 1
TXT      : 0

Sample Files
 - BENIGN_STANDARDIZED.parquet
 - metadata.csv
 - validation_report.csv
 - statistics.json
 - test.parquet
 - validation.parquet
 - train.parquet

CSV DETAILS
----------------------------------------------------------------------
metadata.csv
Rows Loaded : 5
Columns     : 26
Column Names:
   sample_id
   source_benign_id
   text
   label
   source_dataset
   image_path
   runtime_ocr
   attack_family
   attack_name
   attack_location
   attack_type
   visibility
   severity
   ocr_survivability
   target_component
   payload_id
   generation_timestamp
   source_text_hash
   image_width
   image_height
  

In [32]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/TEXT/benchmark_text.csv")

print(df["label"].value_counts())

label
BENIGN    148474
1         144937
Name: count, dtype: int64


In [33]:
print(df["source_dataset"].value_counts())

source_dataset
GENERATED_MALICIOUS          144937
INSURANCE_CSV                139055
INSURANCE_DOCUMENT_IMAGES      7224
PARQUET_DATASET                2195
Name: count, dtype: int64


In [34]:
import pandas as pd

m = pd.read_csv("/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/metadata.csv")

print(m["label"].value_counts())

print(m["source_dataset"].value_counts())

/tmp/ipykernel_2341/823471778.py:3: DtypeWarning: Columns (3,5,7,8,9,10,11,12,13,14,15,16,17,20,21,23) have mixed types. Specify dtype option on import or set low_memory=False.
  m = pd.read_csv("/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/metadata.csv")


label
BENIGN    148474
1          98304
1          46633
Name: count, dtype: int64
source_dataset
GENERATED_MALICIOUS          144937
INSURANCE_CSV                139055
INSURANCE_DOCUMENT_IMAGES      7224
PARQUET_DATASET                2195
Name: count, dtype: int64


In [35]:
print(df["label"].map(type).value_counts())

label
<class 'str'>    293411
Name: count, dtype: int64


In [36]:
from pathlib import Path

img_dir = Path("/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/IMAGES")

imgs = list(img_dir.glob("*"))

print("Total Images :", len(imgs))

print("\nFirst 10 Images:")
for i in imgs[:10]:
    print(i.name)

Total Images : 7303

First 10 Images:
IMG_01037.png
IMG_01038.png
IMG_01039.png
IMG_01040.png
IMG_01041.png
IMG_01042.png
IMG_01043.png
IMG_01044.png
IMG_01045.png
IMG_01046.png


In [37]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/metadata.csv",
    low_memory=False
)

print(df["image_path"].dropna().head(10).tolist())

['/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/IMAGES/IMG_00024.jpg', '/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/IMAGES/IMG_00025.jpg', '/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/IMAGES/IMG_00026.jpg', '/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/IMAGES/IMG_00027.jpg', '/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/IMAGES/IMG_00028.jpg', '/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/IMAGES/IMG_00029.jpg', '/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/IMAGES/IMG_00030.jpg', '/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/IMAGES/IMG_00031.jpg', '/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/IMAGES/IMG_00032.jpg', '/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/IMAGES/IMG_00033.jpg']


In [38]:
from pathlib import Path

paths = df["image_path"].fillna("")

valid = paths.map(lambda x: Path(x).exists() if x else True)

print("Existing Image Paths :", valid.sum())
print("Broken Image Paths   :", (~valid).sum())

Existing Image Paths : 293411
Broken Image Paths   : 0


In [39]:
import os

folder = "/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/IMAGES"

print("Files :", len(os.listdir(folder)))

Files : 7303


In [40]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/metadata.csv",
    low_memory=False
)

broken = df[
    df["image_path"].fillna("").apply(
        lambda x: x != "" and not Path(x).exists()
    )
]

print("Broken:", len(broken))

display(
    broken[
        ["sample_id","source_dataset","image_path"]
    ].head(20)
)

Broken: 0


,sample_id,source_dataset,image_path


In [41]:
from pathlib import Path

broken = df[
    df["image_path"].fillna("").apply(
        lambda x: x != "" and not Path(x).exists()
    )
]

print(broken["image_path"].iloc[0])

IndexError: single positional indexer is out-of-bounds

In [ ]:
from pathlib import Path

p = Path(broken["image_path"].iloc[0])

print("Expected filename:", p.name)

folder = Path("/content/drive/MyDrive/FINAL_BENCHMARK_DATASET/IMAGES")

matches = list(folder.glob(p.stem + ".*"))

print("Possible matches:")

for m in matches:
    print(m.name)

In [ ]:
from pathlib import Path

src = Path("/content/drive/MyDrive/MASTER_BENIGN_DATASET/IMAGE_SOURCES")

print((src/"IMG_02048.png").exists())
print((src/"IMG_02048.jpg").exists())
print((src/"IMG_02048.jpeg").exists())

In [ ]:
mask = df["image_path"].fillna("").apply(
    lambda x: x != "" and not Path(x).exists()
)

df.loc[mask, "image_path"] = ""

In [ ]:
print(df_bimg["image_path"].head(10).tolist())

In [ ]:
print(df_bimg["image_path"].iloc[0])